### RAG Pipelines - Data Ingestion to Vector DB Pipeline


In [1]:

import os
from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pathlib import Path

/var/folders/m3/9ky13j253gb0c5vg13vdzx5w0000gn/T/ipykernel_9740/835167180.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader


In [2]:
### Read all the pdf's inside the directory
def process_all_pdfs(pdf_directory):
    """Process all PDF files in a directory"""
    all_documents = []
    pdf_dir = Path(pdf_directory)
    
    # Find all PDF files recursively
    pdf_files = list(pdf_dir.glob("**/*.pdf"))
    
    print(f"Found {len(pdf_files)} PDF files to process")
    
    for pdf_file in pdf_files:
        print(f"\nProcessing: {pdf_file.name}")
        try:
            loader = PyPDFLoader(str(pdf_file))
            documents = loader.load()
            
            # Add more source information to metadata
            for doc in documents:
                doc.metadata['source_file'] = pdf_file.name
                doc.metadata['file_type'] = 'pdf'
                doc.metadata['file_path'] = str(pdf_file)
            
            all_documents.extend(documents)
            print(f"  ✓ Loaded {len(documents)} pages")
            
        except Exception as e:
            print(f"  ✗ Error: {e}")
    
    print(f"\nTotal documents loaded: {len(all_documents)}")
    return all_documents

# Process all PDFs in the data directory
all_pdf_documents = process_all_pdfs("../data")

Found 1 PDF files to process

Processing: Precalculus4EMB.pdf
  ✓ Loaded 938 pages

Total documents loaded: 938


In [3]:
all_pdf_documents

[Document(metadata={'producer': 'pdfTeX-1.40.20', 'creator': 'LaTeX with hyperref', 'creationdate': '2022-01-04T11:26:11-05:00', 'author': '', 'title': '', 'subject': '', 'keywords': '', 'moddate': '2022-01-04T11:26:11-05:00', 'trapped': '/False', 'ptex.fullbanner': 'This is pdfTeX, Version 3.14159265-2.6-1.40.20 (TeX Live 2019) kpathsea version 6.3.1', 'source': '../data/pdf/Precalculus4EMB.pdf', 'total_pages': 938, 'page': 0, 'page_label': 'i', 'source_file': 'Precalculus4EMB.pdf', 'file_type': 'pdf', 'file_path': '../data/pdf/Precalculus4EMB.pdf'}, page_content='Precalculus\nVersion 4− ϵMB\nby\nCarl Stitz, Ph.D. Jeff Zeager, Ph.D.\nLakeland Community College Lorain County Community College\nJanuary 4, 2022'),
 Document(metadata={'producer': 'pdfTeX-1.40.20', 'creator': 'LaTeX with hyperref', 'creationdate': '2022-01-04T11:26:11-05:00', 'author': '', 'title': '', 'subject': '', 'keywords': '', 'moddate': '2022-01-04T11:26:11-05:00', 'trapped': '/False', 'ptex.fullbanner': 'This is pd

In [4]:
### Text splitting get into chunks

def split_documents(documents,chunk_size=1000,chunk_overlap=200):
    """Split documents into smaller chunks for better RAG performance"""
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        length_function=len,
        separators=["\n\n", "\n", " ", ""]
    )
    split_docs = text_splitter.split_documents(documents)
    print(f"Split {len(documents)} documents into {len(split_docs)} chunks")
    
    # Show example of a chunk
    if split_docs:
        print(f"\nExample chunk:")
        print(f"Content: {split_docs[0].page_content[:200]}...")
        print(f"Metadata: {split_docs[0].metadata}")
    
    return split_docs

In [5]:
chunks = split_documents(all_pdf_documents)

chunks

Split 938 documents into 2201 chunks

Example chunk:
Content: Precalculus
Version 4− ϵMB
by
Carl Stitz, Ph.D. Jeff Zeager, Ph.D.
Lakeland Community College Lorain County Community College
January 4, 2022...
Metadata: {'producer': 'pdfTeX-1.40.20', 'creator': 'LaTeX with hyperref', 'creationdate': '2022-01-04T11:26:11-05:00', 'author': '', 'title': '', 'subject': '', 'keywords': '', 'moddate': '2022-01-04T11:26:11-05:00', 'trapped': '/False', 'ptex.fullbanner': 'This is pdfTeX, Version 3.14159265-2.6-1.40.20 (TeX Live 2019) kpathsea version 6.3.1', 'source': '../data/pdf/Precalculus4EMB.pdf', 'total_pages': 938, 'page': 0, 'page_label': 'i', 'source_file': 'Precalculus4EMB.pdf', 'file_type': 'pdf', 'file_path': '../data/pdf/Precalculus4EMB.pdf'}


[Document(metadata={'producer': 'pdfTeX-1.40.20', 'creator': 'LaTeX with hyperref', 'creationdate': '2022-01-04T11:26:11-05:00', 'author': '', 'title': '', 'subject': '', 'keywords': '', 'moddate': '2022-01-04T11:26:11-05:00', 'trapped': '/False', 'ptex.fullbanner': 'This is pdfTeX, Version 3.14159265-2.6-1.40.20 (TeX Live 2019) kpathsea version 6.3.1', 'source': '../data/pdf/Precalculus4EMB.pdf', 'total_pages': 938, 'page': 0, 'page_label': 'i', 'source_file': 'Precalculus4EMB.pdf', 'file_type': 'pdf', 'file_path': '../data/pdf/Precalculus4EMB.pdf'}, page_content='Precalculus\nVersion 4− ϵMB\nby\nCarl Stitz, Ph.D. Jeff Zeager, Ph.D.\nLakeland Community College Lorain County Community College\nJanuary 4, 2022'),
 Document(metadata={'producer': 'pdfTeX-1.40.20', 'creator': 'LaTeX with hyperref', 'creationdate': '2022-01-04T11:26:11-05:00', 'author': '', 'title': '', 'subject': '', 'keywords': '', 'moddate': '2022-01-04T11:26:11-05:00', 'trapped': '/False', 'ptex.fullbanner': 'This is pd

### Embedding And VectorStoreDB


In [6]:
import numpy as np
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings
import uuid
from typing import List, Dict, Any, Tuple
from sklearn.metrics.pairwise import cosine_similarity

W1030 12:16:50.012000 9740 torch/distributed/elastic/multiprocessing/redirects.py:35] NOTE: Redirects are currently not supported in MacOs.


In [7]:
# document embedding generation using SentenceTransformer

class EmbeddingManager:
    
    def __init__(self, model_name: str = "all-MiniLM-L6-v2"):
        """
        Initialize the embedding manager
        
        Args: model_name: HuggingFace model name for sentence embeddings
        """
        self.model_name = model_name
        self.model = None
        self._load_model()

    def _load_model(self):
        """Load the SentenceTransformer model"""
        try:
            print(f"Loading embedding model: {self.model_name}")
            self.model = SentenceTransformer(self.model_name)
            print(f"Model loaded successfully. Embedding dimension: {self.model.get_embedding_dimension()}")
        except Exception as e:
            print(f"Error loading model {self.model_name}: {e}")
            raise

    def generate_embeddings(self, texts: List[str]) -> np.ndarray:
        """
        Generate embeddings for a list of texts
        
        Args: texts: List of text strings to embed
            
        Returns: numpy array of embeddings with shape (len(texts), embedding_dim)
        """
        if not self.model:
            raise ValueError("Model not loaded")
        
        print(f"Generating embeddings for {len(texts)} texts...")
        embeddings = self.model.encode(texts, show_progress_bar=True)
        print(f"Generated embeddings with shape: {embeddings.shape}")
        return embeddings


## initialize the embedding manager

embedding_manager=EmbeddingManager()
embedding_manager

Loading embedding model: all-MiniLM-L6-v2


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Model loaded successfully. Embedding dimension: 384


### Vector Store


In [8]:
class VectorStore:
    """Manages document embeddings in a ChromaDB vector store"""
    
    def __init__(self, collection_name: str = "pdf_documents", persist_directory: str = "../data/vector_store"):
        """
        Initialize the vector store
        
        Args:
            collection_name: Name of the ChromaDB collection
            persist_directory: Directory to persist the vector store
        """
        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.client = None
        self.collection = None
        self._initialize_store()

    def _initialize_store(self):
        """Initialize ChromaDB client and collection"""
        try:
            # Create persistent ChromaDB client
            os.makedirs(self.persist_directory, exist_ok=True)
            self.client = chromadb.PersistentClient(path=self.persist_directory)
            
            # Get or create collection
            self.collection = self.client.get_or_create_collection(
                name=self.collection_name,
                metadata={"description": "PDF document embeddings for RAG"}
            )
            print(f"Vector store initialized. Collection: {self.collection_name}")
            print(f"Existing documents in collection: {self.collection.count()}")
            
        except Exception as e:
            print(f"Error initializing vector store: {e}")
            raise

    def add_documents(self, documents: List[Any], embeddings: np.ndarray):
        """
        Add documents and their embeddings to the vector store
        
        Args:
            documents: List of LangChain documents
            embeddings: Corresponding embeddings for the documents
        """
        if len(documents) != len(embeddings):
            raise ValueError("Number of documents must match number of embeddings")
        
        print(f"Adding {len(documents)} documents to vector store...")
        
        # Prepare data for ChromaDB
        ids = []
        metadatas = []
        documents_text = []
        embeddings_list = []
        
        for i, (doc, embedding) in enumerate(zip(documents, embeddings)):
            # Generate unique ID
            doc_id = f"doc_{uuid.uuid4().hex[:8]}_{i}"
            ids.append(doc_id)
            
            # Prepare metadata
            metadata = dict(doc.metadata)
            metadata['doc_index'] = i
            metadata['content_length'] = len(doc.page_content)
            metadatas.append(metadata)
            
            # Document content
            documents_text.append(doc.page_content)
            
            # Embedding
            embeddings_list.append(embedding.tolist())
        
        # Add to collection
        try:
            self.collection.add(
                ids=ids,
                embeddings=embeddings_list,
                metadatas=metadatas,
                documents=documents_text
            )
            print(f"Successfully added {len(documents)} documents to vector store")
            print(f"Total documents in collection: {self.collection.count()}")
            
        except Exception as e:
            print(f"Error adding documents to vector store: {e}")
            raise

vectorstore=VectorStore()
vectorstore
    

Vector store initialized. Collection: pdf_documents
Existing documents in collection: 2211


In [9]:
chunks

[Document(metadata={'producer': 'pdfTeX-1.40.20', 'creator': 'LaTeX with hyperref', 'creationdate': '2022-01-04T11:26:11-05:00', 'author': '', 'title': '', 'subject': '', 'keywords': '', 'moddate': '2022-01-04T11:26:11-05:00', 'trapped': '/False', 'ptex.fullbanner': 'This is pdfTeX, Version 3.14159265-2.6-1.40.20 (TeX Live 2019) kpathsea version 6.3.1', 'source': '../data/pdf/Precalculus4EMB.pdf', 'total_pages': 938, 'page': 0, 'page_label': 'i', 'source_file': 'Precalculus4EMB.pdf', 'file_type': 'pdf', 'file_path': '../data/pdf/Precalculus4EMB.pdf'}, page_content='Precalculus\nVersion 4− ϵMB\nby\nCarl Stitz, Ph.D. Jeff Zeager, Ph.D.\nLakeland Community College Lorain County Community College\nJanuary 4, 2022'),
 Document(metadata={'producer': 'pdfTeX-1.40.20', 'creator': 'LaTeX with hyperref', 'creationdate': '2022-01-04T11:26:11-05:00', 'author': '', 'title': '', 'subject': '', 'keywords': '', 'moddate': '2022-01-04T11:26:11-05:00', 'trapped': '/False', 'ptex.fullbanner': 'This is pd

### Convert the text to embeddings

In [10]:
texts = [doc.page_content for doc in chunks]
texts


['Precalculus\nVersion 4− ϵMB\nby\nCarl Stitz, Ph.D. Jeff Zeager, Ph.D.\nLakeland Community College Lorain County Community College\nJanuary 4, 2022',
 'Table of Contents\nThe Old Preface ix\nThe New Preface xiii\n1 Introduction to Functions 1\n1.1 Functions and their Representations . . . . . . . . . . . . . . . . . . . . . . . . . . . . 1\n1.1.1 Functions as Mappings . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . 1\n1.1.2 Algebraic Representations of Functions . . . . . . . . . . . . . . . . . . . . . . . 6\n1.1.3 Geometric Representations of Functions . . . . . . . . . . . . . . . . . . . . . . 10\n1.1.4 Exercises . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . 20\n1.1.5 Answers . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . 30\n1.2 Constant and Linear Functions . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . 40\n1.2.1 Constant Functions . . . . . . . . . . . . . . . . . . . . . . . . . . . . .

In [11]:
texts = [doc.page_content for doc in chunks]

# Generate the embeddings for the chunks

embeddings = embedding_manager.generate_embeddings(texts)

#Store the documents and embeddings in the vector store

vectorstore.add_documents(chunks, embeddings)

Generating embeddings for 2201 texts...


Batches:   0%|          | 0/69 [00:00<?, ?it/s]

Generated embeddings with shape: (2201, 384)
Adding 2201 documents to vector store...
Successfully added 2201 documents to vector store
Total documents in collection: 4412


### Retriever Pipeline From VectorStore

In [12]:
class RAGRetriever:
    """Handles query-based retrieval from the vector store"""
    
    def __init__(self, vector_store: VectorStore, embedding_manager: EmbeddingManager):
        """
        Initialize the retriever
        
        Args:
            vector_store: Vector store containing document embeddings
            embedding_manager: Manager for generating query embeddings
        """
        self.vector_store = vector_store
        self.embedding_manager = embedding_manager

    def retrieve(self, query: str, top_k: int = 5, score_threshold: float = 0.0) -> List[Dict[str, Any]]:
        """
        Retrieve relevant documents for a query
        
        Args:
            query: The search query
            top_k: Number of top results to return
            score_threshold: Minimum similarity score threshold
            
        Returns:
            List of dictionaries containing retrieved documents and metadata
        """
        print(f"Retrieving documents for query: '{query}'")
        print(f"Top K: {top_k}, Score threshold: {score_threshold}")
        
        # Generate query embedding
        query_embedding = self.embedding_manager.generate_embeddings([query])[0]
        
        # Search in vector store
        try:
            results = self.vector_store.collection.query(
                query_embeddings=[query_embedding.tolist()],
                n_results=top_k
            )
            
            # Process results
            retrieved_docs = []
            
            if results['documents'] and results['documents'][0]:
                documents = results['documents'][0]
                metadatas = results['metadatas'][0]
                distances = results['distances'][0]
                ids = results['ids'][0]
                
                for i, (doc_id, document, metadata, distance) in enumerate(zip(ids, documents, metadatas, distances)):
                    # Convert distance to similarity score (ChromaDB uses cosine distance)
                    similarity_score = 1 - distance
                    
                    if similarity_score >= score_threshold:
                        retrieved_docs.append({
                            'id': doc_id,
                            'content': document,
                            'metadata': metadata,
                            'similarity_score': similarity_score,
                            'distance': distance,
                            'rank': i + 1
                        })
                
                print(f"Retrieved {len(retrieved_docs)} documents (after filtering)")
            else:
                print("No documents found")
            
            return retrieved_docs
            
        except Exception as e:
            print(f"Error during retrieval: {e}")
            return []

rag_retriever=RAGRetriever(vectorstore,embedding_manager)
rag_retriever 

In [13]:
rag_retriever.retrieve("Logarithm Function")

Retrieving documents for query: 'Logarithm Function'
Top K: 5, Score threshold: 0.0
Generating embeddings for 1 texts...


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Generated embeddings with shape: (1, 384)
Retrieved 5 documents (after filtering)


[{'id': 'doc_9781a144_1192',
  'content': 'we evaluate f(3) = 23 = 8, the input 3 becomes the exponent on the base 2 to produce the real number 8.\nThe function f−1(x) = log2(x) then takes the number 8 as its input and returns the exponent 3 as its output.\nIn symbols, log2(8) = 3.\nMore generally, log2(x) is the exponent you put on 2 to get x. Thus, log 2(16) = 4, because 2 4 = 16. The\nfollowing theorem summarizes the basic properties of logarithmic functions, all of which come from the fact\nthat they are inverses of exponential functions.',
  'metadata': {'producer': 'pdfTeX-1.40.20',
   'creator': 'LaTeX with hyperref',
   'file_path': '../data/pdf/Precalculus4EMB.pdf',
   'total_pages': 938,
   'keywords': '',
   'doc_index': 1192,
   'ptex.fullbanner': 'This is pdfTeX, Version 3.14159265-2.6-1.40.20 (TeX Live 2019) kpathsea version 6.3.1',
   'moddate': '2022-01-04T11:26:11-05:00',
   'file_type': 'pdf',
   'page': 493,
   'source': '../data/pdf/Precalculus4EMB.pdf',
   'title':

## RAG Pipeline- VectorDB To LLM Output Generation

In [25]:
# Initialize the Gemini LLM

from langchain_google_genai import ChatGoogleGenerativeAI
import os
from dotenv import load_dotenv
load_dotenv()

# Initialize the Gemini LLM
google_api_key = os.getenv("GEMINI_API_KEY")

if not google_api_key:
    raise ValueError("GEMINI_API_KEY not found!")

llm = ChatGoogleGenerativeAI(
    model="gemini-3.8-flash",
    google_api_key=google_api_key,
    temperature=0.1,
    max_tokens=1024
)

# Simple RAG query function that retrieves documents and generates a response using the Gemini LLM

def rag_simple(query, retriever, llm, top_k=3):

    # Retrieve the context
    results = retriever.retrieve(query, top_k=top_k)

    context = "\n\n".join(
        [doc['content'] for doc in results]
    ) if results else ""

    if not context:
        return "No relevant context found to answer the question."

    # Generate the answer using Gemini
    prompt = f"""
    Use the following context to answer the question concisely.

    Context:
    {context}

    Question: {query}

    Answer:
    """

    response = llm.invoke(prompt.format(query=query, context=context))

    return response.content

In [30]:
query = "What is a linear function?"

answer = rag_simple(
    query=query,
    retriever=rag_retriever,
    llm=llm,
    top_k=3
)

answer = answer[0]["text"] if answer else "No answer generated."

print(answer)


Retrieving documents for query: 'What is a linear function?'
Top K: 3, Score threshold: 0.0
Generating embeddings for 1 texts...


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Generated embeddings with shape: (1, 384)
Retrieved 3 documents (after filtering)
Based on the context, a linear function is a function of the form **$f(x) = mx + b$**, where $m$ and $b$ are real numbers with $m \neq 0$. Its domain is $(-\infty, \infty)$, and its graph is the line $y = mx + b$.


## Enhanced RAG Pipeline Features

In [44]:
# --- Enhanced RAG Pipeline Features ---
def rag_advanced(query, retriever, llm, top_k=5, min_score=0.2, return_context=False):
    """
    RAG pipeline with extra features:
    - Returns answer, sources, confidence score, and optionally full context.
    """
    results = retriever.retrieve(query, top_k=top_k, score_threshold=min_score)
    if not results:
        return {'answer': 'No relevant context found.', 'sources': [], 'confidence': 0.0, 'context': ''}
    
    # Prepare context and sources
    context = "\n\n".join([doc['content'] for doc in results])
    sources = [{
        'source': doc['metadata'].get('source_file', doc['metadata'].get('source', 'unknown')),
        'page': doc['metadata'].get('page', 'unknown'),
        'score': doc['similarity_score'],
        'preview': doc['content'][:300] + '...'
    } for doc in results]
    confidence = max([doc['similarity_score'] for doc in results])
    
    # Generate answer
    prompt = f"""Use the following context to answer the question concisely.\nContext:\n{context}\n\nQuestion: {query}\n\nAnswer:"""
    response = llm.invoke([prompt.format(context=context, query=query)])
    
    output = {
        'answer': response.content,
        'sources': sources,
        'confidence': confidence
    }
    if return_context:
        output['context'] = context
    return output

# Example usage:
result = rag_advanced("What is a linear function?", rag_retriever, llm, top_k=3, min_score=0, return_context=True)
print("Answer:", result['answer'])
print("Sources:", result['sources'])
print("Confidence:", result['confidence'])
print("Context Preview:", result['context'][:300])

Retrieving documents for query: 'What is a linear function?'
Top K: 3, Score threshold: 0
Generating embeddings for 1 texts...


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Generated embeddings with shape: (1, 384)
Retrieved 3 documents (after filtering)
Answer: [{'type': 'text', 'text': 'Based on the context, a linear function is a function of the form **$f(x) = mx + b$**, where $m$ and $b$ are real numbers with $m \\neq 0$, having a domain of $(-\\infty, \\infty)$ and a graph representing the line $y = mx + b$.', 'extras': {'signature': 'EvcJCvQJAWkUfRNJio6ONBXQztFun19zX2toJ2fTF6YEDybiGjNsAi/KxZVuZrXUQm5t2PZ0ve/YIG21GCn3Azm+9oqhuaVAYtEca+OCE2yHAdwf12F4Rae0hanUA3/7Fy/nB/Hwket2BtpopEmCMOa/TrFw6iEFDu+kF18ZYIMAoqynjcvz1vJMTYxYg1g/4wly09JvzbqAObG45pIG3vnJSj0dgFAxCiIE8yq2fcHqfcvNNUfz+pjXQcgJLEH/jtoDr0vqQcolVXDHnSeIim2kzzy35HRYKBg3j4qqbBWnAskLYYWCAHbhaSsN6hOXZryiAy4nvLvtKynect2JhC3FlnVhGZqNtkDMC7VIhdK4Jdh3NPoyFeSu5wqOUz9tuSv/6E3N8AZEbaeal3PHDOhGwi15chYPGftrLpby8yJp8G1R9KW2fjuncHyjE21jWftarAkwNqREt8eRL0NiGlEHd9CPX7+lUc4XBf6NOGPN9/eXplGv5egxUzk8g38sT6iuTvwq+gbbGu5Q3FEkxlCEhFS9vSSbAyx/D3LD+TrePn188ZbVOfQM1buptffJi6vXssg82V9h9mLA0Z8hrKvIYi0SQ6oN84ibVitRxSRaMDtLsR9

In [43]:
# --- Advanced RAG Pipeline: Streaming, Citations, History, Summarization ---
from typing import List, Dict, Any
import time

class AdvancedRAGPipeline:
    def __init__(self, retriever, llm):
        self.retriever = retriever
        self.llm = llm
        self.history = []  # Store query history

    def query(self, question: str, top_k: int = 5, min_score: float = 0.2, stream: bool = False, summarize: bool = False) -> Dict[str, Any]:
        # Retrieve relevant documents
        results = self.retriever.retrieve(question, top_k=top_k, score_threshold=min_score)
        if not results:
            answer = "No relevant context found."
            sources = []
            context = ""
        else:
            context = "\n\n".join([doc['content'] for doc in results])
            sources = [{
                'source': doc['metadata'].get('source_file', doc['metadata'].get('source', 'unknown')),
                'page': doc['metadata'].get('page', 'unknown'),
                'score': doc['similarity_score'],
                'preview': doc['content'][:120] + '...'
            } for doc in results]
            # Streaming answer simulation
            prompt = f"""Use the following context to answer the question concisely.\nContext:\n{context}\n\nQuestion: {question}\n\nAnswer:"""
            if stream:
                print("Streaming answer:")
                for i in range(0, len(prompt), 80):
                    print(prompt[i:i+80], end='', flush=True)
                    time.sleep(0.05)
                print()
            response = self.llm.invoke([prompt.format(context=context, question=question)])
            answer = response.content[0]["text"]

        # Add citations to answer
        citations = [f"[{i+1}] {src['source']} (page {src['page']})" for i, src in enumerate(sources)]
        answer_with_citations = answer + "\n\nCitations:\n" + "\n".join(citations) if citations else answer

        # Optionally summarize answer
        summary = None
        if summarize and answer:
            summary_prompt = f"Summarize the following answer in 2 sentences:\n{answer}"
            summary_resp = self.llm.invoke([summary_prompt])
            summary = summary_resp.content

        # Store query history
        self.history.append({
            'question': question,
            'answer': answer,
            'sources': sources,
            'summary': summary
        })

        return {
            'question': question,
            'answer': answer_with_citations,
            'sources': sources,
            'summary': summary,
            'history': self.history
        }

# Example usage:
adv_rag = AdvancedRAGPipeline(rag_retriever, llm)
result = adv_rag.query("what is a linear function?", top_k=3, min_score=0.05, stream=True, summarize=True)
print("\nFinal Answer:", result['answer'])
print("Summary:", result['summary'])
print("History:", result['history'][-1])

Retrieving documents for query: 'what is a linear function?'
Top K: 3, Score threshold: 0.05
Generating embeddings for 1 texts...


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Generated embeddings with shape: (1, 384)
Retrieved 2 documents (after filtering)
Streaming answer:
Use the following context to answer the question concisely.
Context:
where m and b are real numbers with m⁄= 0. The domain of a linear function is (−∞,∞).
As with Deﬁnition 1.4, in Deﬁnition 1.5, x is the independent variable, f is the function name, and both m
and b are parameters. Notice that m is restricted by m⁄= 0 for if m = 0 then the function f(x) = mx +b would
reduce to the constant function f(x) = b. The domain of linear functions, like that of constant functions, is
speciﬁed as (−∞,∞)
Recall6 that the form of the line y = mx + b is called the slope-intercept form of the line and the slope, m,
and the y-intercept (0, b), are easily determined when the line is written this way. Likewise, the form of the
function in Deﬁnition 1.5, f(x) = mx + b, is often called the slope-intercept form of a linear function.
The graph of a linear function is the graph of the line y = mx + b. Lines 